In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any, TypedDict, List, Optional

from langgraph.graph import StateGraph, START, END
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

from pydantic import BaseModel, Field
import os


from core import *

True


In [2]:
from core import LF_CONFIG, LLM_BASIC, RETRY

LF_CONFIG

{'callbacks': [<langfuse.langchain.CallbackHandler.LangchainCallbackHandler at 0x73a556ce6a10>]}

In [3]:
import langchain
langchain.__version__

'1.2.13'

In [4]:
from core import *

In [5]:


class SourceDataState(BaseModel):
    workspace_path: str
    key: str
    list_files: List[str] = Field(default_factory=list)
    description_files: List[str] = Field(default_factory=list)
    data_files: List[str] = Field(default_factory=list)
    other_files: List[str] = Field(default_factory=list)

class BatchSourceDataState(BaseModel):
    workspace_path: str
    keys: List[str]
    results: Annotated[List[dict], operator.add] = Field(default_factory=list)


In [6]:

detect_files_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify files into dataset (CSV), description (TXT/MD/BIB), and other."),
    ("user", """
Return JSON with:
- data_files
- description_files
- other_files

Rules:
- Only use given names
- No edits, no inventions
- [] if none

Files:
{file_list}
""".strip()),
])


class DetectedFiles(BaseModel):
    description_files: List[str] = Field(default_factory=list)
    data_files: List[str] = Field(default_factory=list)
    other_files: List[str] = Field(default_factory=list)


def validate_files_exist(workspace_path: str, key: str, filelist_all: list[str]) -> list[str]:
    root = Path(workspace_path) / key
    return [f for f in filelist_all if not (root / f).exists()]


def _invoke_with_backoff(structured_llm, messages, max_retries: int = 5):
    import time

    for attempt in range(max_retries + 1):
        try:
            return structured_llm.invoke(messages)
        except Exception as e:
            message = str(e)
            is_rate_limit = ("429" in message) or ("RateLimitError" in message)
            if (not is_rate_limit) or (attempt == max_retries):
                raise
            time.sleep(min(2 ** attempt, 16))


def detect_candidate_files(state: SourceDataState):
    structured_llm = LLM_BASIC.with_structured_output(DetectedFiles)

    file_list_text = "\n".join(state.list_files)

    messages = detect_files_prompt.format_messages(file_list=file_list_text)
    result = _invoke_with_backoff(structured_llm, messages)

    # Reflection and retry
    for _ in range(RETRY):
        missing_files = validate_files_exist(
            state.workspace_path,
            state.key,
            result.description_files + result.data_files,
        )

        if not missing_files:
            break

        messages = messages + [
            ("ai", result.model_dump_json()),
            ("user", f"These files do not exist: {missing_files}. Fix the JSON."),
        ]

        result = _invoke_with_backoff(structured_llm, messages)

    return {
        "description_files": result.description_files,
        "data_files": result.data_files,
        "other_files": result.other_files,
    }

In [7]:
def list_files_in_folder(state: SourceDataState):
    root = Path(state.workspace_path) / state.key

    files = sorted(
        p.name
        for p in root.glob("*")
        if p.is_file()
    )

    return {"list_files": files}


def store_state(state: SourceDataState):
    key_folder = Path(state.workspace_path) / state.key
    VersionedYaml(key_folder).create("detect_files", state.model_dump())

In [9]:
from pathlib import Path
from typing import List, Annotated
import operator

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from core import LF_CONFIG


# ---------- subgraph: process one key ----------

subgraph = StateGraph(SourceDataState)
subgraph.add_node("node_1", list_files_in_folder)
subgraph.add_node("node_2", detect_candidate_files)
subgraph.add_node("node_3", store_state)

subgraph.add_edge(START, "node_1")
subgraph.add_edge("node_1", "node_2")
subgraph.add_edge("node_2", "node_3")
subgraph.add_edge("node_3", END)

source_data_app = subgraph.compile()


# ---------- parent graph: process many keys ----------

class BatchSourceDataState(BaseModel):
    workspace_path: str
    keys: List[str]
    results: Annotated[List[dict], operator.add] = Field(default_factory=list)


def dispatch_keys(state: BatchSourceDataState):
    return [
        Send(
            "process_one_key",
            {
                "workspace_path": state.workspace_path,
                "key": key,
            },
        )
        for key in state.keys
    ]


def process_one_key(state):
    result = source_data_app.invoke(
        {
            "workspace_path": state["workspace_path"],
            "key": state["key"],
        }
    )
    return {"results": [result]}


graph = StateGraph(BatchSourceDataState)
graph.add_node("process_one_key", process_one_key)

graph.add_conditional_edges(START, dispatch_keys)
graph.add_edge("process_one_key", END)

graph2 = graph.compile()


# ---------- run ----------
keys = sorted(
    p.name
    for p in Path(WORKSPACE).iterdir()
    if p.is_dir() and not p.name.startswith(".")
)

initial_state = {
    "workspace_path": WORKSPACE,
    "keys": keys,
}

run_config = {**LF_CONFIG, "max_concurrency": 6}
result = graph2.invoke(initial_state, config=run_config)

In [11]:
d,_ = VersionedYaml(Path(WORKSPACE) / "ALGALTOX").read("detect_files")
d

{'workspace_path': '/home/kostanew/2026/CDGReview/workspace',
 'key': 'ALGALTOX',
 'list_files': ['ALGALTOX_14022019.csv',
  'ALGALTOX_InChIKeys_14022019.txt',
  'README.md',
  'chemicals.csv'],
 'description_files': ['ALGALTOX_InChIKeys_14022019.txt', 'README.md'],
 'data_files': ['ALGALTOX_14022019.csv', 'chemicals.csv'],
 'other_files': []}